In [5]:
import jax
import jax.numpy as jnp
from jax.scipy.linalg import expm

# Attempt to import logm from jax.scipy.linalg, falling back to jax.experimental.linalg
try:
    from jax.scipy.linalg import logm
except ImportError:
    try:
        from jax.experimental.linalg import logm
    except ImportError:
        raise ImportError("Could not import logm from jax.scipy.linalg or jax.experimental.linalg. Please check your JAX installation.")

# Enable 64-bit precision for "proof-grade" norm checks
jax.config.update("jax_enable_x64", True)

def get_diagonal_kinetic(dim, g_sq):
    """
    Kinetic term: -d^2/dtheta^2  --> n^2 in Fourier basis
    This corresponds to the Electric Field energy (E^2).
    """
    n = jnp.arange(-(dim - 1) // 2, (dim - 1) // 2 + 1)
    # H_kin = (g^2 / 2) * n^2
    return jnp.diag((g_sq / 2.0) * n**2)

def get_potential_interaction(dim, lambda_c):
    """
    Potential term: -cos(theta_i - theta_{i+1})
    Magnetic Field energy (B^2).
    Off-diagonal in Fourier basis (hopping by +/- 1).
    """
    # Create shift matrices
    off_diag = jnp.ones(dim - 1)
    # Cosine = (Shift_Up + Shift_Down) / 2
    # lambda_c is 1/g^2 (inverse coupling)
    V = -lambda_c * (jnp.diag(off_diag, k=1) + jnp.diag(off_diag, k=-1)) / 2.0
    return V

def build_block_hamiltonian(dim, g_sq, lambda_c, sites=2):
    """
    Builds H for a block of 'sites' size.
    For 2 sites: H = K1 + K2 + V_12
    """
    K = get_diagonal_kinetic(dim, g_sq)
    I = jnp.eye(dim)

    # H_local = K x I + I x K + Interaction
    H_kin = jnp.kron(K, I) + jnp.kron(I, K)

    # Interaction acts on both
    # We explicitly build the operator cos(theta_1 - theta_2)
    # In basis |n1, n2>, this couples |n1, n2> to |n1+/-1, n2-/+1>
    # This preserves total angular momentum N = n1 + n2 (Global Gauge Invariance)

    # Constructing V_12 operator in tensor product space
    shift = jnp.diag(jnp.ones(dim - 1), k=1)
    shift_dag = jnp.diag(jnp.ones(dim - 1), k=-1)

    # e^{i(t1 - t2)} = e^{it1} e^{-it2}
    term1 = jnp.kron(shift, shift_dag)
    term2 = jnp.kron(shift_dag, shift)

    H_pot = -lambda_c * (term1 + term2) / 2.0

    return H_kin + H_pot

def projector_onto_gauge_invariant(dim):
    """
    The Projector Pi_{a -> a'}.
    We map the block of 2 sites to 1 effective site.
    Strategy: Keep the lowest energy states of the relative motion,
    preserving the total charge sector.
    """
    # Simply summing states is naive.
    # Real Space RG: We keep the lowest eigenmodes of the block Hamiltonian.
    return jnp.eye(dim**2) # Placeholder for the identity (no truncation for pure norm check first)

@jax.jit
def run_defect_check(a_fine):
    """
    THE ALGORITHM:
    1. Construct Fine Hamiltonian H_a (scale a)
    2. Compute T_fine = exp(-a * H_a)
    3. Block: T_blocked = T_fine * T_fine (Time doubling a -> 2a)
    4. Extract K_eff = -log(T_blocked) / (2a)
    5. Construct Target H_2a (scale 2a, renormalized coupling)
    6. Compute Defect || K_eff - H_2a ||
    """

    dim = 7  # Truncation of Fourier modes

    # SCALING LAWS (Asymptotic Freedom / Continuum Limit)
    # For 4D YM, g^2 ~ 1 / log(1/a).
    # For this 1D model (QM), scaling is simpler, but we simulate the critical approach.
    # Let's assume canonical scaling for checking: g -> g (invariant in 1D) or g -> g/sqrt(2)

    # Initial Parameters at scale 'a'
    g_fine = 1.0
    lambda_fine = 1.0 / (g_fine**2)

    # 1. H_fine (2-site block)
    H_fine = build_block_hamiltonian(dim, g_fine**2, lambda_fine, sites=2)

    # 2. Transfer Matrix T_fine = exp(-a * H)
    T_fine = expm(-a_fine * H_fine)

    # 3. Block: Coarse Graining in Time (a -> 2a)
    # We also coarse grain in space (2 sites -> 1 site) for the full check,
    # but here we check the generator consistency first.
    T_blocked = T_fine @ T_fine

    # 4. Extract Effective Generator
    # K_eff = -(1/2a) * log(T_blocked)
    K_eff = -logm(T_blocked) / (2 * a_fine)

    # 5. Target Generator (Canonical H at scale 2a)
    # Is the physics at 2a just the same Hamiltonian?
    # If the theory is scale invariant (critical), yes.
    # If massive, we expect deviations.
    # Let's assume the target is the SAME Hamiltonian structure (perfect RG fixed point).
    H_target = H_fine

    # 6. Norm Difference (The Defect)
    # We focus on the low-energy sector (center of matrix)
    defect_matrix = K_eff - H_target
    norm_diff = jnp.linalg.norm(defect_matrix)

    return norm_diff.real

# RUN THE CHECK
print(f"{'Lattice (a)':<15} | {'Defect ||K_eff - H||':<25} | {'Convergence?'}")
print("-" * 60)

# We check a sequence of lattice spacings approaching the continuum (a -> 0)
spacings = [0.1, 0.05, 0.01, 0.005, 0.001]

for a in spacings:
    delta = run_defect_check(a)
    status = "CONVERGING" if delta < 1.0 else "DIVERGING"
    print(f"{a:<15.4f} | {delta:<25.6e} | {status}")

ImportError: Could not import logm from jax.scipy.linalg or jax.experimental.linalg. Please check your JAX installation.